In [1]:
!pip install google-genai pydantic

import os
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

In [2]:
from google.colab import userdata

# Puxa a chave API_KEY cadastrada nos Secrets do Colab
api_key = userdata.get('API_KEY')

# Inicializa o cliente do Gemini
client = genai.Client(api_key=api_key)


In [3]:
# PASSO 2: Criar a função que consulta os dados do chip Ollie Pet (Sprint 2)

def consultar_telemetria_ollie(pet_id: str) -> dict:
    """Busca em tempo real as métricas do chip IoT Ollie Pet (ESP32).

    Retorna a temperatura corporal (°C), batimentos cardíacos (BPM),
    distância da residência em metros e o status do alerta (LED/Buzzer).
    """
    # Simulando os dados exatos do seu circuito do vídeo (com a febre de 45°C)
    return {
        "pet_name": "Ollie",
        "temperatura_c": 45.0,        # Simulação do teste de febre/alerta do vídeo
        "batimentos_bpm": 125,        # Batimentos dentro da faixa normal
        "distancia_metros": 12,        # Distância em relação à casa
        "status_alerta_hardware": True # Indica que a luz de alerta e o som/buzzer apitaram
    }

In [4]:
# ==========================================
# PASSO 3: Prompt do Sistema (System Instruction)
# ==========================================
system_instruction = """
Você é o assistente virtual de saúde da clínica veterinária CLYVO VET, integrado ao dispositivo IoT Ollie Pet.

Suas responsabilidades e regras:
1. Sempre que o tutor perguntar sobre a saúde, temperatura, batimentos ou localização do pet, invoque a ferramenta `consultar_telemetria_ollie`.
2. Parâmetros clínicos de referência:
   - Temperatura corporal: Faixa normal entre 38.0°C e 39.2°C. Valores acima de 39.5°C indicam febre/hipertermia.
   - Frequência cardíaca (BPM): Faixa normal de 70 a 130 BPM para cães de médio porte.
   - Perímetro de segurança: Até 50 metros da residência.
3. Tratamento de Alertas e Emergências:
   - Se a temperatura estiver em 45°C ou o `status_alerta_hardware` for True, acione imediatamente um ALERTA DE EMERGÊNCIA.
   - Explique amigavelmente ao tutor que a temperatura está muito elevada (45.0°C), que o alarme do chip foi disparado e recomende levar o Ollie imediatamente à clínica CLYVO VET ou oferecer primeiros socorros (água fresca, local ventilado).
4. Mantenha um tom profissional, empático e acolhedor.
"""

# ==========================================
# PASSO 4: Criar o Chat com Tool Calling (Gemini)
# ==========================================
from google.genai import types

# Inicializa o chat conectando as instruções e a função do chip
chat = client.chats.create(
    model="gemini-3.6-flash",
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        tools=[consultar_telemetria_ollie],
        temperature=0.3
    )
)

# ==========================================
# PASSO 5: Função de Conversa e Testes Práticos
# ==========================================
def conversar(mensagem_tutor: str):
    print(f"🐶 Tutor: {mensagem_tutor}")
    resposta = chat.send_message(mensagem_tutor)
    print(f"🤖 CLYVO VET IA:\n{resposta.text}\n")
    print("-" * 60)

# Executando os testes:
conversar("Olá! O que o chip Ollie Pet monitora?")
conversar("Como estão a temperatura e os batimentos do Ollie agora? Ele tá bem?")

🐶 Tutor: Olá! O que o chip Ollie Pet monitora?
🤖 CLYVO VET IA:
Olá! Seja muito bem-vindo(a) à **CLYVO VET**! 🐾

O dispositivo IoT **Ollie Pet** (embarcado com tecnologia ESP32) é uma ferramenta avançada de monitoramento em tempo real da saúde e segurança do seu pet. Ele monitora:

1. **🌡️ Temperatura Corporal (°C):** Acompanha a temperatura do pet para identificar precocemente febre, hipertermia ou hipotermia (faixa normal de 38.0°C a 39.2°C).
2. **💓 Frequência Cardíaca (BPM):** Mede os batimentos cardíacos por minuto para monitorar o esforço físico e a saúde cardiovascular (faixa normal de 70 a 130 BPM para cães de médio porte).
3. **📍 Localização e Perímetro de Segurança:** Calcula a distância em metros em relação à sua residência (com limite de segurança de até 50 metros), ajudando a evitar fugas ou perdas.
4. **🚨 Sistema de Alerta Integrado (LED/Buzzer):** Dispara alarmes visuais e sonoros no próprio dispositivo em situações críticas de saúde ou alteração brusca dos sinais vitais.
